### **Hypothesis Testing Fundamentals**

In [ ]:
# z-score: how many standard deviations your observed result is away from what the                                                                        null hypothesis says should happen. "z=(x−μ)/σ"​
# NOTE: To calculate it, you need three numbers: 
#     1. the sample statistic (point estimate)
#     2. the hypothesized statistic
#     3. the standard error of the statistic (estimated from the bootstrap distribution).

mean_comp_samp = DF['var'].mean() 
std_error = np.std(DF_boot_distn, ddof=1)    # This comes from a bootstrap distribution
mean_comp_hyp = 0.06                         # EXAMPLE: 6%
z_score = (mean_samp - mean_hyp) / std_error 
------------------------------------------------------------------------------------------
# In order to determine whether to choose the null hypothesis or the alternative                                                             hypothesis, you need to calculate a p-value from the z-score.


# H0 (NULL hypothesis): prop. of data scientists starting programming as children (35%)
# HA (ALTERNATIVE hypothesis):                      different/more than/ less than 35%

# The test ends in: 1. "reject H0"
#                   2. "fail to reject H0" --------------------------------->>> p-value
# Significance level: "beyond a reasonable doubt" for hypothesis testing

![My Chart](../images/19.png)

![My Chart](../images/20.png)

In [ ]:
norm.cdf()                                           # Left-tailed test
1 - norm.cdf()                                       # Right-tailed test

# Significance level of a hypothesis test (α): the threshold point for a reasonable doubt
# Common values of "α": 0.2, 0.1, 0.05, 0.01 (should be set prior to conducting experiment

# "p ≤ α" :  "reject H0" , else fail to reject H0

from scipy.stats import norm                     # for this example
alpha = 0.05
p_value = 1 - norm.cdf(z_score, loc=0, scale=1)
p_value <= alpha                                 # THE FINAL RESULT OF HYPOTHESIS TESTING

# Confidence intervals: For a significance level (α), the confidence interval level of 1-α
lower = np.quantile(DF_boot_distn, 0.025)        # alpha = 0.05 : ci = 95%
upper = np.quantile(DF_boot_distn, 0.975)

![My Chart](../images/21.png)

### **Two-Sample and ANOVA Tests**

In [ ]:
# Two-sample problems: Compare sample statistics across groups of a variable. 
# Example: Are users who first programmed as a child compensated higher than those                                                                          that started as adults?: H0:μ(child)−μ(adult)=0, HA:μ(child)−μ(adult)>0
xbar = stack_overflow.groupby('age_first_code_cut')['converted_comp'].mean() 
s = stack_overflow.groupby('age_first_code_cut')['converted_comp'].std() 
n = stack_overflow.groupby('age_first_code_cut')['converted_comp'].count() 

numerator = xbar_child - xbar_adult 
denominator = np.sqrt(s_child ** 2 / n_child + s_adult ** 2 / n_adult) 
t_statistics = numerator / denominator
# T-statistics HAS THE SAME LOGIC OF z-score BUT FOR 2 CATEGORIES OF A VARIABLE

from scipy.stats import t 
degrees_of_freedom = n_child + n_adult - 2 
1 - t.cdf(t_statistics, df=degrees_of_freedom)

![My Chart](../images/22.png)

In [ ]:
# For Categories
# Paired: Two datasets are paired when each observation in one dataset has a specific,                                                               meaningful partner in the other dataset. 
# (like the percentage of Republican candidate votes in 2008 and 2012)
import pingouin                                          # using pingouin is easier method
pingouin.ttest(x=DF['repub_2008'], y=DF['repub_2012'], paired=True, alternative="less")
# alternative is "two-sided", "less", or "greater" based on the problems.

# VIP NOTE: Unpaired t-tests on paired data increases the chances of false negative errors
------------------------------------------------------------------------------------------
alpha = 0.2 
# Analysis of variance (ANOVA)
pingouin.anova(data=DF, dv="converted_comp", between="job_sat")
#                       dependent variable
pingouin.pairwise_tests(data=DF, dv="converted_comp", between="job_sat", padjust="none")

pingouin.pairwise_tests(data=DF, dv="converted_comp", between="job_sat", padjust="bonf") 

![My Chart](../images/23.png)

### **Proportion Tests**

In [ ]:
#  p: population proportion (unknown population parameter)
p_hat = (DF['age_cat'] == 'Under 30').mean()  # p^: sample proportion (sample statistic)
p_0 = 0.50                                    # P0: hypothesized population proportion
n = len(DF) 
numerator = p_hat - p_0 
denominator = np.sqrt(p_0 * (1 - p_0) / n) 
z_score = numerator / denominator                               # SEE THE FORMULA
p_value = 2 * (1 - norm.cdf(z_score))                           # Two-tailed
------------------------------------------------------------------------------------------
# proportions_ztest() for 2 samples
DF.groupby("age_cat")['hobbyist'].value_counts()
n_hobbyists = np.array([812, 1021]) 
n_rows = np.array([812 + 238, 1021 + 190]) 
from statsmodels.stats.proportion import proportions_ztest 
z_score, p_value = proportions_ztest(count=n_hobbyists, nobs=n_rows, 
                                     alternative="two-sided")
------------------------------------------------------------------------------------------
# Statistical Independence: proportion of successes in the response variable is the same                                                       across all categories of the explanatory variable (chi2 used for DEPENDENCY problems)
import pingouin 
expected, observed, stats = pingouin.chi2_independence(data=DF, x='var1', y='var2', 
                                                       correction=False) 

# NOTE: Like the t-distribution, the chi-square distribution has degrees of freedom and                                                              non-centrality parameters. When these numbers are large, the chi-square distribution can                                                                  be approximated by a normal distribution.
degree_of_freedom = (No_of_response_categories − 1) × (No_of_explanatory_categories − 1)

# NOTE2: The chi-square test statistic is a square number, so it is always non-negative,                                                           so only the "right-tailed" tends to be of interest.
------------------------------------------------------------------------------------------
# chi-square goodness of fit test: compares proportions of each level of a categorical                                                      variable (observed) to hypothesized values. 
from scipy.stats import chisquare 
chisquare(f_obs=observed['n'], f_exp=hypothesized['n'])

### **Non-Parametric Tests**

In [ ]:
# FUNDAMENTAL ASSUMPTIONS IN HYPOTHESIS TESTING
# 1. "Randomness" Assumption: The samples are random subsets of larger populations.
# 2. "Independence" Assumption: Each observation (row) in the dataset is independent.
# 3. "Large sample size" Assumption: The sample is big enough to mitigate uncertainty,                                                                         so that the  Central Limit Theorem applies. 
#                                   t-test: n>= 30, 
#                                   proportion tests: n*p-hat >= 10, n*(1-p-hat)>=10
#                                   chi-square tests: n*p-hat >= 5, n*(1-p-hat)>=5

# Sanity check: If the bootstrap distribution doesn't look normal, assumptions likely are                                                         not valid. Revisit data collection to check for randomness, independence, and sample size
------------------------------------------------------------------------------------------
#  Non-parametric tests: more reliable than parametric tests for "SMALL" sample sizes and                                                      when data isn't normally distributed.

# Wilcoxon (or "rank test": for PAIRED variables) - You can do it manually too
pingouin.wilcoxon(x=DF['var1'], y=DF['var2'], alternative="...")

# Wilcoxon-Mann-Whitney (or "rank sum test":for UNPAIRED)
var1_vs_var2 = DF[['var1', 'var2']] 
var1_vs_var2_wide = age_vs_comp.pivot(columns='var2', values='var1')
pingouin.mwu(x=var1_vs_var2_wide['var1'], y=var1_vs_var2_wide['var2'], alternative='...')

# Kruskal-Wallis (non-parametric version of an ANOVA test)
pingouin.kruskal(data=stack_overflow, dv='converted_comp', between='job_sat')